
# Teacher Regularization v1.0

`baseline v2.1`를 기준 backbone으로 사용하고, `DANN v1.0`의 domain adaptation loss를 참고해 teacher regularization을 추가한 notebook입니다.

구성:
- student: baseline v2.1의 `MultiViewBidirectionalCrossAttention`
- teacher target group 1: `physics`
- teacher target group 2: `image_structure`
- auxiliary: `domain_head` with GRL


In [1]:

from __future__ import annotations

import os
import sys
import json
import random
import shutil
from dataclasses import dataclass
from itertools import cycle
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
import timm

SRC_DIR = (Path.cwd() / '../../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from output_paths import allocate_output_paths
from reproducibility import make_generator, seed_everything, seed_worker
from models import (
    EMAConfig,
    ModelEMA,
    MultiViewBidirectionalCrossAttentionConfig,
)

DATA_DIR = (Path.cwd() / '../../data').resolve()
FEATURE_CSV = (Path.cwd() / '../../outputs/physics_feature_analysis_v2/class_analysis_features.csv').resolve()
assert DATA_DIR.exists(), f'data dir not found: {DATA_DIR}'
assert FEATURE_CSV.exists(), f'feature csv not found: {FEATURE_CSV}'
print('DATA_DIR:', DATA_DIR)
print('FEATURE_CSV:', FEATURE_CSV)

PHYSICS_FEATURES = [
    'top_fill_ratio',
    'top_centroid_dx',
    'front_centroid_dx',
    'front_tilt',
    'top_support_width_frac',
]
IMAGE_STRUCTURE_FEATURES = [
    'abs_delta_structure_center_x',
    'top_structure_center_x',
    'front_structure_center_x',
    'mean_structure_center_x',
    'mean_structure_bbox_w',
]
ALL_TEACHER_FEATURES = PHYSICS_FEATURES + IMAGE_STRUCTURE_FEATURES

CFG = {
    'IMG_SIZE': 320,
    'EPOCHS': 70,
    'LEARNING_RATE': 1e-4,
    'BATCH_SIZE': 8,
    'SEED': 42,
    'NUM_WORKERS': 8,
    'WEIGHT_DECAY': 1e-4,
    'MIN_LR': 1e-6,
    'EMA_DECAY': 0.999,
    'EMA_USE_FOR_EVAL': True,
    'EARLY_STOPPING_PATIENCE': 8,
    'MIXUP_ALPHA': 0.1,
    'MIXUP_PROB': 0.0,
    'VIDEO_AUG_ENABLE': True,
    'VIDEO_AUG_CACHE': True,
    'UNSTABLE_START_MIN_SEC': 0.5,
    'UNSTABLE_START_MAX_SEC': 1.0,
    'UNSTABLE_FRAMES_MIN': 2,
    'UNSTABLE_FRAMES_MAX': 3,
    'STABLE_END_MIN_SEC': 9.0,
    'STABLE_END_MAX_SEC': 10.0,
    'STABLE_FRAMES_PER_VIDEO': 2,
    'BACKBONE_NAME': 'efficientnetv2_rw_m',
    'BACKBONE_GRAD_CHECKPOINTING': True,
    'ATTN_DIM': 256,
    'NUM_HEADS': 8,
    'NUM_LAYERS': 2,
    'POS_GRID': 7,
    'DROPOUT': 0.1,
    'CLASSIFIER_HIDDEN_DIM': 512,
    'CLASSIFIER_MID_DIM': 128,
    'CLASSIFIER_DROPOUT': 0.3,
    'DOMAIN_HIDDEN_DIM': 256,
    'DOMAIN_DROPOUT': 0.2,
    'PHYSICS_LOSS_WEIGHT': 0.15,
    'IMAGE_LOSS_WEIGHT': 0.15,
    'DOMAIN_LOSS_WEIGHT': 0.2,
    'GRL_WARMUP_EPOCHS': 5,
    'GRL_MAX_LAMBDA': 0.2,
    'GRL_GAMMA': 4.0,
}

seed_everything(CFG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
amp_enabled = (device.type == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
print('device:', device)


DATA_DIR: /media/hdd0/whyz/structure-stability/data
FEATURE_CSV: /media/hdd0/whyz/structure-stability/outputs/physics_feature_analysis_v2/class_analysis_features.csv
device: cuda


/tmp/ipykernel_2766456/1472265973.py:112: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)


In [2]:

train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
dev_df = pd.read_csv(DATA_DIR / 'dev.csv', encoding='utf-8-sig')
test_df = pd.read_csv(DATA_DIR / 'sample_submission.csv', encoding='utf-8-sig')
feature_df = pd.read_csv(FEATURE_CSV, encoding='utf-8-sig')

feature_df = feature_df[['split', 'sample_id'] + ALL_TEACHER_FEATURES].copy()
feature_df = feature_df.rename(columns={'sample_id': 'id'})

for split_name, split_df in [('train', train_df), ('dev', dev_df)]:
    part = feature_df[feature_df['split'].eq(split_name)].drop(columns=['split']).copy()
    missing = set(split_df['id']) - set(part['id'])
    print(split_name, 'teacher rows:', len(part), 'missing:', len(missing))

physics_scaler = StandardScaler()
image_scaler = StandardScaler()
train_phys_mask = feature_df['split'].eq('train') & feature_df[PHYSICS_FEATURES].notna().all(axis=1)
train_img_mask = feature_df['split'].eq('train') & feature_df[IMAGE_STRUCTURE_FEATURES].notna().all(axis=1)
physics_scaler.fit(feature_df.loc[train_phys_mask, PHYSICS_FEATURES])
image_scaler.fit(feature_df.loc[train_img_mask, IMAGE_STRUCTURE_FEATURES])

all_phys_mask = feature_df[PHYSICS_FEATURES].notna().all(axis=1)
all_img_mask = feature_df[IMAGE_STRUCTURE_FEATURES].notna().all(axis=1)
feature_df.loc[all_phys_mask, PHYSICS_FEATURES] = physics_scaler.transform(feature_df.loc[all_phys_mask, PHYSICS_FEATURES])
feature_df.loc[all_img_mask, IMAGE_STRUCTURE_FEATURES] = image_scaler.transform(feature_df.loc[all_img_mask, IMAGE_STRUCTURE_FEATURES])

print('teacher target normalization: StandardScaler fitted on train split')
print('physics normalized rows:', int(all_phys_mask.sum()), 'image normalized rows:', int(all_img_mask.sum()))

train_df = train_df.merge(feature_df[feature_df['split'].eq('train')].drop(columns=['split']), on='id', how='left')
dev_df = dev_df.merge(feature_df[feature_df['split'].eq('dev')].drop(columns=['split']), on='id', how='left')
test_df['sample_dir'] = str(DATA_DIR / 'test')

print('train:', train_df.shape)
print('dev:', dev_df.shape)


train teacher rows: 1000 missing: 0
dev teacher rows: 100 missing: 0
teacher target normalization: StandardScaler fitted on train split
physics normalized rows: 1100 image normalized rows: 1100
train: (1000, 12)
dev: (100, 12)


In [3]:

def _extract_frame_by_sec(cap, sec, fps, frame_count):
    frame_idx = int(max(0, min(frame_count - 1, round(sec * fps))))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _extract_last_frame(cap, frame_count):
    last_idx = max(0, frame_count - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, last_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _video_aug_cache_signature(cfg):
    keys = [
        'SEED',
        'UNSTABLE_START_MIN_SEC',
        'UNSTABLE_START_MAX_SEC',
        'UNSTABLE_FRAMES_MIN',
        'UNSTABLE_FRAMES_MAX',
        'STABLE_END_MIN_SEC',
        'STABLE_END_MAX_SEC',
        'STABLE_FRAMES_PER_VIDEO',
    ]
    return {k: cfg.get(k) for k in keys}


def build_video_augmented_df(train_df, data_dir, cfg):
    train_root = data_dir / 'train'
    aug_root = data_dir / 'train_video_aug'
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / 'aug_df.csv'
    cache_meta = aug_root / 'cache_meta.json'
    cache_sig = _video_aug_cache_signature(cfg)

    if cfg.get('VIDEO_AUG_CACHE', True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get('signature') == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    cached_df['sample_dir'] = str(aug_root)
                    return cached_df
        except Exception as exc:
            print('video aug cache read failed:', exc)

    for p in aug_root.glob('AUGV_*'):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(cfg['SEED'])
    saved_idx = 0
    aug_rows = []

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f'AUGV_{saved_idx:07d}'
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / 'front.png')
        Image.fromarray(img).save(out_dir / 'top.png')
        row = {'id': aug_id, 'label': label, 'sample_dir': str(aug_root)}
        for col in ALL_TEACHER_FEATURES:
            row[col] = np.nan
        aug_rows.append(row)
        saved_idx += 1

    for row in tqdm(train_df.itertuples(index=False), total=len(train_df), desc='video aug', dynamic_ncols=True, ascii=True):
        sample_id = row.id
        label = row.label
        video_path = train_root / sample_id / 'simulation.mp4'
        if not video_path.exists():
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue

        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps <= 0 or frame_count <= 0:
            cap.release()
            continue

        if label == 'unstable':
            n_frames = int(rng.integers(cfg['UNSTABLE_FRAMES_MIN'], cfg['UNSTABLE_FRAMES_MAX'] + 1))
            secs = rng.uniform(cfg['UNSTABLE_START_MIN_SEC'], cfg['UNSTABLE_START_MAX_SEC'], size=n_frames)
            for sec in secs:
                frame = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
                if frame is not None:
                    save_aug(frame, label)
        else:
            n_frames = int(cfg['STABLE_FRAMES_PER_VIDEO'])
            secs = rng.uniform(cfg['STABLE_END_MIN_SEC'], cfg['STABLE_END_MAX_SEC'], size=n_frames)
            for sec in secs:
                frame = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
                if frame is not None:
                    save_aug(frame, label)

        cap.release()

    aug_df = pd.DataFrame(aug_rows)
    aug_df.to_csv(cache_csv, index=False)
    cache_meta.write_text(json.dumps({'signature': cache_sig}, indent=2))
    return aug_df


class TeacherRegularizationDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}
        self.ids = self.df['id'].astype(str).tolist()
        self.sample_dirs = self.df['sample_dir'].astype(str).tolist() if 'sample_dir' in self.df.columns else None
        if not self.is_test:
            self.labels = [self.label_map[v] for v in self.df['label'].astype(str).tolist()]
            self.physics_targets = self.df[PHYSICS_FEATURES].to_numpy(dtype=np.float32)
            self.image_targets = self.df[IMAGE_STRUCTURE_FEATURES].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = self.ids[idx]
        base_dir = self.sample_dirs[idx] if self.sample_dirs is not None else self.root_dir
        folder_path = os.path.join(base_dir, sample_id)

        views = []
        for name in ['front', 'top']:
            img_path = os.path.join(folder_path, f'{name}.png')
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return {'views': views, 'id': sample_id}

        label = self.labels[idx]
        physics_target = self.physics_targets[idx]
        image_target = self.image_targets[idx]
        return {
            'views': views,
            'label': torch.tensor(label, dtype=torch.float32),
            'physics_target': torch.tensor(physics_target, dtype=torch.float32),
            'image_target': torch.tensor(image_target, dtype=torch.float32),
            'has_teacher': torch.tensor(float(np.isfinite(physics_target).all() and np.isfinite(image_target).all()), dtype=torch.float32),
            'id': sample_id,
        }


train_transform, test_transform = build_default_transforms(CFG['IMG_SIZE'])
train_df_for_train = train_df.copy()
train_df_for_train['sample_dir'] = str(DATA_DIR / 'train')
if CFG['VIDEO_AUG_ENABLE']:
    aug_df = build_video_augmented_df(train_df, DATA_DIR, CFG)
    if not aug_df.empty:
        train_df_for_train = pd.concat([train_df_for_train, aug_df], ignore_index=True)

dev_domain_df = dev_df.copy()
dev_domain_df['sample_dir'] = str(DATA_DIR / 'dev')
dev_eval_df = dev_df.copy()
dev_eval_df['sample_dir'] = str(DATA_DIR / 'dev')

train_dataset = TeacherRegularizationDataset(train_df_for_train, str(DATA_DIR / 'train'), train_transform, is_test=False)
dev_domain_dataset = TeacherRegularizationDataset(dev_domain_df, str(DATA_DIR / 'dev'), train_transform, is_test=False)
dev_eval_dataset = TeacherRegularizationDataset(dev_eval_df, str(DATA_DIR / 'dev'), test_transform, is_test=False)
test_dataset = TeacherRegularizationDataset(test_df, str(DATA_DIR / 'test'), test_transform, is_test=True)

loader_kwargs = dict(
    batch_size=CFG['BATCH_SIZE'],
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'),
    persistent_workers=(CFG['NUM_WORKERS'] > 0),
)
if CFG['NUM_WORKERS'] > 0:
    loader_kwargs['prefetch_factor'] = 2
train_loader = DataLoader(train_dataset, shuffle=True, worker_init_fn=seed_worker, generator=make_generator(CFG['SEED']), **loader_kwargs)
dev_domain_loader = DataLoader(dev_domain_dataset, shuffle=True, worker_init_fn=seed_worker, generator=make_generator(CFG['SEED'] + 1), **loader_kwargs)
dev_eval_loader = DataLoader(dev_eval_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print('train_dataset:', len(train_dataset))
print('dev_domain_dataset:', len(dev_domain_dataset))
print('dev_eval_dataset:', len(dev_eval_dataset))
print('test_dataset:', len(test_dataset))


train_dataset: 3000
dev_domain_dataset: 100
dev_eval_dataset: 100
test_dataset: 1000


In [4]:

@dataclass(frozen=True)
class TeacherRegularizationConfig:
    backbone_name: str = CFG['BACKBONE_NAME']
    pretrained: bool = True
    attn_dim: int = CFG['ATTN_DIM']
    num_heads: int = CFG['NUM_HEADS']
    num_layers: int = CFG['NUM_LAYERS']
    pos_grid: int = CFG['POS_GRID']
    dropout: float = CFG['DROPOUT']
    classifier_hidden_dim: int = CFG['CLASSIFIER_HIDDEN_DIM']
    classifier_mid_dim: int = CFG['CLASSIFIER_MID_DIM']
    classifier_dropout: float = CFG['CLASSIFIER_DROPOUT']
    domain_hidden_dim: int = CFG['DOMAIN_HIDDEN_DIM']
    domain_dropout: float = CFG['DOMAIN_DROPOUT']
    physics_dim: int = len(PHYSICS_FEATURES)
    image_dim: int = len(IMAGE_STRUCTURE_FEATURES)
    out_dim: int = 1


class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradientReversalFunction.apply(x, lambda_)


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

    def forward(self, q_tokens, kv_tokens):
        q = self.norm_q(q_tokens)
        kv = self.norm_kv(kv_tokens)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        return q_tokens + attn_out


class TeacherRegularizedMultiView(nn.Module):
    def __init__(self, config: TeacherRegularizationConfig | None = None):
        super().__init__()
        self.config = config or TeacherRegularizationConfig()
        self.backbone = timm.create_model(
            self.config.backbone_name,
            pretrained=self.config.pretrained,
            num_classes=0,
            global_pool='',
        )
        if CFG.get('BACKBONE_GRAD_CHECKPOINTING', False) and hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(True)
        feature_dim = self.backbone.num_features
        self.proj = nn.Conv2d(feature_dim, self.config.attn_dim, kernel_size=1, bias=False)
        self.pos_embed = nn.Parameter(torch.randn(1, self.config.attn_dim, self.config.pos_grid, self.config.pos_grid) * 0.02)
        self.cross_12 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.cross_21 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.classifier = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim),
            nn.BatchNorm1d(self.config.classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.classifier_dropout),
            nn.Linear(self.config.classifier_hidden_dim, self.config.classifier_mid_dim),
            nn.ReLU(),
            nn.Linear(self.config.classifier_mid_dim, self.config.out_dim),
        )
        self.physics_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.physics_dim),
        )
        self.image_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.image_dim),
        )
        self.domain_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.domain_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.domain_dropout),
            nn.Linear(self.config.domain_hidden_dim, 1),
        )

    def _to_tokens(self, feat_map):
        feat_map = self.proj(feat_map)
        pos = F.interpolate(self.pos_embed, size=feat_map.shape[-2:], mode='bilinear', align_corners=False)
        feat_map = feat_map + pos
        return feat_map.flatten(2).transpose(1, 2)

    def extract_features(self, views):
        f1 = self.backbone.forward_features(views[0])
        f2 = self.backbone.forward_features(views[1])
        t1 = self._to_tokens(f1)
        t2 = self._to_tokens(f2)
        for blk12, blk21 in zip(self.cross_12, self.cross_21):
            t1_prev, t2_prev = t1, t2
            t1 = blk12(t1_prev, t2_prev)
            t2 = blk21(t2_prev, t1_prev)
        return torch.cat([t1.mean(dim=1), t2.mean(dim=1)], dim=1)

    def forward_domain(self, views, lambda_=0.0):
        feat = self.extract_features(views)
        dom_feat = grad_reverse(feat, lambda_)
        return self.domain_head(dom_feat).view(-1)

    def forward(self, views, lambda_=0.0):
        feat = self.extract_features(views)
        out = {
            'class_logit': self.classifier(feat).view(-1),
            'physics_pred': self.physics_head(feat),
            'image_pred': self.image_head(feat),
            'feat': feat,
        }
        out['domain_logit'] = self.domain_head(grad_reverse(feat, lambda_)).view(-1)
        return out


In [5]:

def mixup_multiview_batch(views, labels, alpha=0.2):
    if alpha <= 0:
        return views, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = labels.size(0)
    index = torch.randperm(batch_size, device=labels.device)
    mixed_views = [lam * v + (1.0 - lam) * v[index, :] for v in views]
    return mixed_views, labels, labels[index], lam


def compute_lambda(epoch_idx, step_idx, steps_per_epoch, total_epochs, warmup_epochs=0, max_lambda=1.0, gamma=10.0):
    total_steps = max(1, total_epochs * steps_per_epoch)
    current_step = max(0, (epoch_idx - 1) * steps_per_epoch + step_idx)
    progress = current_step / total_steps
    warmup_progress = warmup_epochs / max(1, total_epochs)
    if progress <= warmup_progress:
        return 0.0
    effective_progress = (progress - warmup_progress) / max(1e-8, 1.0 - warmup_progress)
    lambda_base = 2.0 / (1.0 + np.exp(-gamma * effective_progress)) - 1.0
    return float(max_lambda * lambda_base)


def masked_smooth_l1(pred, target):
    mask = torch.isfinite(target).all(dim=1)
    if mask.any():
        return F.smooth_l1_loss(pred[mask], target[mask])
    return pred.sum() * 0.0


def train_one_epoch(model, train_loader, dev_loader, optimizer, device, epoch_idx, total_epochs, scaler, ema=None):
    model.train()
    dev_iter = cycle(dev_loader)
    total_rows = []
    steps = len(train_loader)
    amp_enabled = scaler.is_enabled()
    for step_idx, batch in enumerate(tqdm(train_loader, desc='Training', dynamic_ncols=True, ascii=True), start=1):
        dev_batch = next(dev_iter)
        train_views = [v.to(device, non_blocking=True) for v in batch['views']]
        train_labels = batch['label'].to(device, non_blocking=True).float()
        train_phys = batch['physics_target'].to(device, non_blocking=True)
        train_img = batch['image_target'].to(device, non_blocking=True)
        dev_views = [v.to(device, non_blocking=True) for v in dev_batch['views']]

        optimizer.zero_grad(set_to_none=True)

        if CFG['MIXUP_ALPHA'] > 0 and np.random.rand() < CFG['MIXUP_PROB']:
            mixed_views, labels_a, labels_b, lam = mixup_multiview_batch(train_views, train_labels, CFG['MIXUP_ALPHA'])
            with torch.amp.autocast('cuda', enabled=amp_enabled):
                outputs = model(mixed_views, lambda_=0.0)
            loss_cls = lam * F.binary_cross_entropy_with_logits(outputs['class_logit'], labels_a) + (1.0 - lam) * F.binary_cross_entropy_with_logits(outputs['class_logit'], labels_b)
            loss_phys = outputs['physics_pred'].sum() * 0.0
            loss_img = outputs['image_pred'].sum() * 0.0
        else:
            with torch.amp.autocast('cuda', enabled=amp_enabled):
                outputs = model(train_views, lambda_=0.0)
                loss_cls = F.binary_cross_entropy_with_logits(outputs['class_logit'], train_labels)
                loss_phys = masked_smooth_l1(outputs['physics_pred'], train_phys)
                loss_img = masked_smooth_l1(outputs['image_pred'], train_img)

        domain_views = [torch.cat([tv, dv], dim=0) for tv, dv in zip(train_views, dev_views)]
        domain_labels = torch.cat([
            torch.zeros(train_views[0].size(0), device=device),
            torch.ones(dev_views[0].size(0), device=device),
        ], dim=0)
        grl_lambda = compute_lambda(
            epoch_idx,
            step_idx - 1,
            steps,
            total_epochs,
            warmup_epochs=CFG['GRL_WARMUP_EPOCHS'],
            max_lambda=CFG['GRL_MAX_LAMBDA'],
            gamma=CFG['GRL_GAMMA'],
        )
        with torch.amp.autocast('cuda', enabled=amp_enabled):
            domain_logit = model.forward_domain(domain_views, lambda_=grl_lambda)
            loss_dom = F.binary_cross_entropy_with_logits(domain_logit, domain_labels)

        loss = loss_cls + CFG['PHYSICS_LOSS_WEIGHT'] * loss_phys + CFG['IMAGE_LOSS_WEIGHT'] * loss_img + CFG['DOMAIN_LOSS_WEIGHT'] * loss_dom
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update(model)

        with torch.no_grad():
            domain_probs = torch.sigmoid(domain_logit)
            domain_acc = ((domain_probs > 0.5) == domain_labels).float().mean().item()

        total_rows.append({
            'loss': float(loss.item()),
            'loss_cls': float(loss_cls.item()),
            'loss_phys': float(loss_phys.item()),
            'loss_img': float(loss_img.item()),
            'loss_dom': float(loss_dom.item()),
            'domain_acc': float(domain_acc),
        })

    hist = pd.DataFrame(total_rows)
    return hist.mean().to_dict()


@torch.no_grad()
def evaluate_classification(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []
    physics_losses = []
    image_losses = []
    amp_enabled = (device.type == 'cuda')
    for batch in tqdm(loader, desc='Dev Eval', dynamic_ncols=True, ascii=True):
        views = [v.to(device, non_blocking=True) for v in batch['views']]
        labels = batch['label'].to(device, non_blocking=True).float()
        physics_target = batch['physics_target'].to(device, non_blocking=True)
        image_target = batch['image_target'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=amp_enabled):
            outputs = model(views, lambda_=0.0)
            probs = torch.sigmoid(outputs['class_logit'])
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        physics_losses.append(float(masked_smooth_l1(outputs['physics_pred'], physics_target).item()))
        image_losses.append(float(masked_smooth_l1(outputs['image_pred'], image_target).item()))

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    p = np.clip(all_probs, 1e-15, 1 - 1e-15)
    logloss = float(-np.mean(all_labels * np.log(p) + (1.0 - all_labels) * np.log(1.0 - p)))
    acc = float(np.mean((all_probs > 0.5) == all_labels))
    auc = float(__import__('sklearn.metrics').metrics.roc_auc_score(all_labels, all_probs))
    return {
        'dev_logloss': logloss,
        'dev_acc': acc,
        'dev_auc': auc,
        'dev_phys_loss': float(np.mean(physics_losses)),
        'dev_img_loss': float(np.mean(image_losses)),
    }


@torch.no_grad()
def predict_test_probs(model, loader, device):
    model.eval()
    probs_all = []
    ids = []
    amp_enabled = (device.type == 'cuda')
    for batch in tqdm(loader, desc='Test Inference', dynamic_ncols=True, ascii=True):
        views = [v.to(device, non_blocking=True) for v in batch['views']]
        with torch.amp.autocast('cuda', enabled=amp_enabled):
            outputs = model(views, lambda_=0.0)
            probs = torch.sigmoid(outputs['class_logit'])
        probs_all.extend(probs.cpu().numpy())
        ids.extend(batch['id'])
    return ids, np.array(probs_all, dtype=np.float64)


In [6]:

model = TeacherRegularizedMultiView(TeacherRegularizationConfig()).to(device)
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'], eta_min=CFG['MIN_LR'])
ema = ModelEMA(model, EMAConfig(decay=CFG['EMA_DECAY']))

artifacts = allocate_output_paths(experiment_name='teacher_regularization', major_version='v1.5')
best_model_path = artifacts['weight_path']
submission_path = artifacts['submission_path']
history_path = (Path.cwd() / '../../outputs/eda_preprocessing/teacher_regularization_v1.5_history.csv').resolve()
history_path.parent.mkdir(parents=True, exist_ok=True)
print('Artifact version:', artifacts['version'])
print('best_model_path:', best_model_path)
print('submission_path:', submission_path)

best_logloss = float('inf')
best_epoch = -1
patience_left = CFG['EARLY_STOPPING_PATIENCE']
history = []

for epoch in range(1, CFG['EPOCHS'] + 1):
    train_metrics = train_one_epoch(model, train_loader, dev_domain_loader, optimizer, device, epoch, CFG['EPOCHS'], scaler, ema=ema)
    eval_model = ema.ema_model if CFG['EMA_USE_FOR_EVAL'] else model
    dev_metrics = evaluate_classification(eval_model, dev_eval_loader, device)

    improved = dev_metrics['dev_logloss'] < best_logloss
    if improved:
        best_logloss = dev_metrics['dev_logloss']
        best_epoch = epoch
        patience_left = CFG['EARLY_STOPPING_PATIENCE']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'ema_state_dict': ema.ema_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'cfg': CFG,
            **dev_metrics,
        }, best_model_path)
    else:
        patience_left -= 1

    scheduler.step()
    row = {
        'epoch': epoch,
        **{k: float(v) for k, v in train_metrics.items()},
        **dev_metrics,
        'lr': float(optimizer.param_groups[0]['lr']),
        'improved': bool(improved),
        'patience_left': int(patience_left),
    }
    history.append(row)
    print(row)
    if patience_left < 0:
        print('early stop:', epoch)
        break

history_df = pd.DataFrame(history)
history_df.to_csv(history_path, index=False)
print('saved history:', history_path)


Artifact version: v1.5.0
best_model_path: /media/hdd0/whyz/structure-stability/outputs/weights/teacher_regularization_v1.5.0.pt
submission_path: /media/hdd0/whyz/structure-stability/outputs/submissions/teacher_regularization_v1.5.0.csv


Training:   0%|          | 0/375 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:54: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
/tmp/ipykernel_2766456/3503322369.py:74: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Dev Eval:   0%|          | 0/13 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:115: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Dev Eval: 100%|##########| 13/13 [00:02<00:00,  5.61it/s]


{'epoch': 1, 'loss': 0.48174073298772174, 'loss_cls': 0.32137347834308944, 'loss_phys': 0.2544442359923851, 'loss_img': 0.2705780000484859, 'loss_dom': 0.4080695740779241, 'domain_acc': 0.8565555574099223, 'dev_logloss': 0.6903680495622291, 'dev_acc': 0.48, 'dev_auc': 0.7550080128205128, 'dev_phys_loss': 2.0983699835263767, 'dev_img_loss': 0.7209374835857978, 'lr': 9.995015679379509e-05, 'improved': True, 'patience_left': 8}


Training:   0%|          | 0/375 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:54: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
/tmp/ipykernel_2766456/3503322369.py:74: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Dev Eval:   0%|          | 0/13 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:115: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Dev Eval: 100%|##########| 13/13 [00:00<00:00, 17.50it/s]


{'epoch': 2, 'loss': 0.2473009882469972, 'loss_cls': 0.13454265418524544, 'loss_phys': 0.20193327244681616, 'loss_img': 0.22820601745229213, 'loss_dom': 0.24118718659877778, 'domain_acc': 0.9176666682561239, 'dev_logloss': 0.6240082904384988, 'dev_acc': 0.69, 'dev_auc': 0.9497195512820513, 'dev_phys_loss': 2.1097078048265896, 'dev_img_loss': 0.7237395426401725, 'lr': 9.980072755276436e-05, 'improved': True, 'patience_left': 8}


...

{'epoch': 33, 'loss': 0.17327048995097477, 'loss_cls': 0.02387937867711298, 'loss_phys': 0.037106185763763885, 'loss_img': 0.02450855905485029, 'loss_dom': 0.7007444783051808, 'domain_acc': 0.5710555565357208, 'dev_logloss': 0.16326240447938734, 'dev_acc': 0.96, 'dev_auc': 0.9927884615384616, 'dev_phys_loss': 2.024937868118286, 'dev_img_loss': 0.7156663216077365, 'lr': 5.4937145790719963e-05, 'improved': False, 'patience_left': 1}


Training:   0%|          | 0/375 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:54: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
/tmp/ipykernel_2766456/3503322369.py:74: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Dev Eval:   0%|          | 0/13 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:115: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Dev Eval: 100%|##########| 13/13 [00:00<00:00, 17.48it/s]


{'epoch': 34, 'loss': 0.16898838152488072, 'loss_cls': 0.01914063268182023, 'loss_phys': 0.028642029999755323, 'loss_img': 0.02154166480592297, 'loss_dom': 0.7116009573141734, 'domain_acc': 0.540166668176651, 'dev_logloss': 0.17333580320339145, 'dev_acc': 0.96, 'dev_auc': 0.9887820512820512, 'dev_phys_loss': 2.0238643517861, 'dev_img_loss': 0.7126918297547561, 'lr': 5.27208091023505e-05, 'improved': False, 'patience_left': 0}


Training:   0%|          | 0/375 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:54: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
/tmp/ipykernel_2766456/3503322369.py:74: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Dev Eval:   0%|          | 0/13 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:115: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Dev Eval: 100%|##########| 13/13 [00:00<00:00, 17.03it/s]

{'epoch': 35, 'loss': 0.1672272388935089, 'loss_cls': 0.021959636199986563, 'loss_phys': 0.029871393183867136, 'loss_img': 0.018682212830521166, 'loss_dom': 0.6899227938652038, 'domain_acc': 0.5737777791420619, 'dev_logloss': 0.1693749951179275, 'dev_acc': 0.96, 'dev_auc': 0.9871794871794871, 'dev_phys_loss': 2.015308847794166, 'dev_img_loss': 0.7119333503338007, 'lr': 5.050000000000001e-05, 'improved': False, 'patience_left': -1}
early stop: 35
saved history: /media/hdd0/whyz/structure-stability/outputs/eda_preprocessing/teacher_regularization_v1.5_history.csv


In [7]:

checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
best_state = checkpoint['ema_state_dict'] if CFG['EMA_USE_FOR_EVAL'] else checkpoint['model_state_dict']
model.load_state_dict(best_state)
final_dev_metrics = evaluate_classification(model, dev_eval_loader, device)
print('best_epoch:', best_epoch)
print(final_dev_metrics)

ids, test_probs = predict_test_probs(model, test_loader, device)
submission = pd.DataFrame({
    'id': ids,
    'unstable_prob': test_probs,
    'stable_prob': 1.0 - test_probs,
})
submission.to_csv(submission_path, index=False, encoding='utf-8-sig')
print('saved submission:', submission_path)
submission.head()


Dev Eval:   0%|          | 0/13 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:115: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Dev Eval: 100%|##########| 13/13 [00:00<00:00, 18.14it/s]


best_epoch: 26
{'dev_logloss': 0.0730849234640861, 'dev_acc': 0.99, 'dev_auc': 0.9959935897435896, 'dev_phys_loss': 2.0611336689728956, 'dev_img_loss': 0.6712184800551488}


Test Inference:   0%|          | 0/125 [00:00<?, ?it/s]/tmp/ipykernel_2766456/3503322369.py:146: FutureWarning: `torch.amp.autocast('cuda', args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.amp.autocast('cuda', enabled=amp_enabled):
Test Inference: 100%|##########| 125/125 [00:15<00:00,  8.31it/s]

saved submission: /media/hdd0/whyz/structure-stability/outputs/submissions/teacher_regularization_v1.5.0.csv


,id,unstable_prob,stable_prob
0,TEST_0001,0.000082,0.999918
1,TEST_0002,1.000000,0.000000
2,TEST_0003,0.998535,0.001465
3,TEST_0004,0.996582,0.003418
4,TEST_0005,0.370361,0.629639
